# Titanic Model: Logistic Regression

> This module loads cleaned-up data from the Kaggle Titanic ML Competition and model with logistic regression

In [1]:
#| default_exp titanic_logiReg

In [2]:
#| export
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
# %matplotlib inline

import os # interact with system directories and files
import wandb # log data and models with Weights and Biases

In [3]:
#| export
dataPath = '/Users/danc/Data/titanic'
for dirname, _, filenames in os.walk(dataPath):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/Users/danc/Data/titanic/test.csv
/Users/danc/Data/titanic/test_cleaned.csv
/Users/danc/Data/titanic/train_cleaned.csv
/Users/danc/Data/titanic/train.csv
/Users/danc/Data/titanic/gender_submission.csv


In [11]:
#| export
train_data = pd.read_csv(os.path.join(dirname, 'train_cleaned.csv'))
train_data.head()
#len(train_data)

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,male,Q,S
0,1,0,3,22.0,1,0,7.2500,True,False,True
1,2,1,1,38.0,1,0,71.2833,False,False,False
2,3,1,3,26.0,0,0,7.9250,False,False,True
3,4,1,1,35.0,1,0,53.1000,False,False,True
4,5,0,3,35.0,0,0,8.0500,True,False,True


In [12]:
#| export
test_data = pd.read_csv(os.path.join(dirname, 'test_cleaned.csv'))
test_data.head()
#len(test_data)

,PassengerId,Pclass,Age,SibSp,Parch,Fare,male,Q,S
0,892,3,34.5,0,0,7.8292,True,True,False
1,893,3,47.0,1,0,7.0000,False,False,True
2,894,2,62.0,0,0,9.6875,True,True,False
3,895,3,27.0,0,0,8.6625,True,False,True
4,896,3,22.0,1,1,12.2875,False,False,True


In [19]:
#| export
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [20]:
#| export
X_train, X_dev, y_train, y_dev = train_test_split(train_data.drop('Survived',axis=1), 
                                                    train_data['Survived'], test_size=0.30, 
                                                    random_state=101)

In [25]:
#| hide
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 623 entries, 520 to 863
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  623 non-null    int64  
 1   Pclass       623 non-null    int64  
 2   Age          623 non-null    float64
 3   SibSp        623 non-null    int64  
 4   Parch        623 non-null    int64  
 5   Fare         623 non-null    float64
 6   male         623 non-null    bool   
 7   Q            623 non-null    bool   
 8   S            623 non-null    bool   
dtypes: bool(3), float64(2), int64(4)
memory usage: 35.9 KB


In [26]:
#| export
sc = StandardScaler()
X_train = sc.fit_transform(X_train[:,:6])
X_dev = sc.transform(X_dev[:,:6])

InvalidIndexError: (slice(None, None, None), slice(None, 6, None))

In [16]:
#| export
logmodel = LogisticRegression()


In [17]:
#| export
logmodel.fit(X_train,y_train)

/opt/anaconda3/envs/nbenv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [ ]:
#| export
predDev = logmodel.predict(X_dev)
print(classification_report(y_dev,predDev))

In [ ]:
#| export
predTest = logmodel.predict(test_data)
output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predTest})
output.to_csv(os.path.join(dirname, 'logiReg_submission.csv'),index=False)


In [ ]:
#| export
wandb.login() # log in Weights and Biases to upload and log data

In [ ]:
#| export
run = wandb.init(project="Kaggle_Titanic", config={}, job_type="add-dataset")
artifact = wandb.Artifact(name="Titanic_data", type="dataset")
artifact.add_dir(local_path=dataPath)  # Add dataset directory to artifact
run.log_artifact(artifact)  # Logs the artifact version "Titanic_data:v0"
run.finish()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()